# 🎓 Hybrid University Recommendation Engine (v2.0)

This notebook implements a **Two-Layer Hybrid Approach** with **Dynamic Language Switching**:
1.  **Rule-Based Filtering**: Checks eligibility. If country is 'Germany', it checks **German Score**; otherwise, it checks **IELTS**.
2.  **Machine Learning Ranking**: Uses a Random Forest model to predict student 'Caliber' and ranks colleges using a country-specific **Match Score**.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

# 📌 Configuration
DATA_PATH = "university_dataset_80_rows.csv"
MODEL_DIR = "saved_models"

if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)

# Load Dataset
df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df.shape[0]} universities available.")

## 🛠️ Step 1: Feature Engineering & ML Training

In [ ]:
le_field = LabelEncoder()
le_degree = LabelEncoder()
le_country = LabelEncoder()
le_intake = LabelEncoder()

df_encoded = df.copy()
df_encoded["Course_Field"] = le_field.fit_transform(df["Course_Field"])
df_encoded["Degree_Level"] = le_degree.fit_transform(df["Degree_Level"])
df_encoded["Country"] = le_country.fit_transform(df["Country"])
df_encoded["Intake"] = le_intake.fit_transform(df["Intake"])

features = [
    "Min_IELTS", "Min_German_Score", "Min_Academic_Percentage",
    "Tuition_Fee_USD", "Backlogs_Allowed", "Work_Experience_Required",
    "Course_Field", "Degree_Level", "Country", "Intake"
]

X = df_encoded[features]
y = df["Ranking_Tier"]

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

# Save artifacts for Django
joblib.dump(model, os.path.join(MODEL_DIR, "university_model.joblib"))
joblib.dump(le_field, os.path.join(MODEL_DIR, "le_field.joblib"))
joblib.dump(le_degree, os.path.join(MODEL_DIR, "le_degree.joblib"))
joblib.dump(le_country, os.path.join(MODEL_DIR, "le_country.joblib"))
joblib.dump(le_intake, os.path.join(MODEL_DIR, "le_intake.joblib"))
joblib.dump(features, os.path.join(MODEL_DIR, "feature_names.joblib"))

print("✅ ML Model trained and artifacts saved.")

## 🔬 Step 2: Dynamic Hybrid Recommendation Logic

In [ ]:
def recommend_universities(student_profile):
    country_choice = student_profile["country"]
    
    # --- LAYER 1: ELIGIBILITY FILTERING ---
    # Common filters
    base_mask = (
        (df["Min_Academic_Percentage"].astype(float) <= float(student_profile["academic_percent"])) &
        (df["Tuition_Fee_USD"].astype(float) <= float(student_profile["budget"])) &
        (df["Backlogs_Allowed"].astype(int) >= int(student_profile["backlogs"])) &
        (df["Course_Field"] == student_profile["field"]) &
        (df["Degree_Level"] == student_profile["degree"])
    )
    
    # Dynamic Language filter
    if country_choice.lower() == "germany":
        lang_mask = (df["Min_German_Score"].astype(float) <= float(student_profile["german_score"])) & (df["Country"] == "Germany")
    else:
        lang_mask = (df["Min_IELTS"].astype(float) <= float(student_profile["ielts"])) & (df["Country"] == country_choice)
    
    eligible_unis = df[base_mask & lang_mask].copy()
    
    if eligible_unis.empty:
        return f"No matching universities found in {country_choice}.", None
        
    # --- LAYER 2: ML CALIBER PREDICTION ---
    student_input = pd.DataFrame([[
        student_profile['ielts'], student_profile['german_score'], student_profile['academic_percent'],
        student_profile['budget'], student_profile['backlogs'], student_profile['work_exp'],
        le_field.transform([student_profile['field']])[0],
        le_degree.transform([student_profile['degree']])[0],
        le_country.transform([student_profile['country']])[0],
        le_intake.transform([student_profile['intake']])[0]
    ]], columns=features)
    
    caliber = model.predict(student_input)[0]

    # --- LAYER 3: SUITABILITY RANKING ---
    if country_choice.lower() == "germany":
        lang_surplus = (float(student_profile["german_score"]) - eligible_unis["Min_German_Score"].astype(float)) * 0.5
    else:
        lang_surplus = (float(student_profile["ielts"]) - eligible_unis["Min_IELTS"].astype(float)) * 10

    eligible_unis["Match_Score"] = (
        lang_surplus +
        (float(student_profile["academic_percent"]) - eligible_unis["Min_Academic_Percentage"].astype(float)) +
        (float(student_profile["work_exp"]) - eligible_unis["Work_Experience_Required"].astype(float)) * 5
    )
    
    ranked_unis = eligible_unis.sort_values(by="Match_Score", ascending=False)
    return caliber, ranked_unis

## 🏆 Step 3: Test Dynamic Logic
Try changing the country to 'Germany' or 'USA' to see the difference.

In [ ]:
test_profile = {
    "ielts": 0, 
    "german_score": 80, 
    "academic_percent": 82,
    "budget": 15000, 
    "backlogs": 0, 
    "work_exp": 1,
    "field": "Engineering", 
    "degree": "Master", 
    "country": "Germany", 
    "intake": "Fall"
}

caliber, recommendations = recommend_universities(test_profile)

if recommendations is not None:
    print(f"🔥 Predicted Student Caliber: {caliber}")
    display(recommendations[["University_Name", "Country", "Min_German_Score", "Match_Score"]].head(5))
else:
    print(caliber)